#  E-Commerce Data Cleaning — Portfolio Project

**Dataset:** East Africa Orders | **5,115 rows × 21 columns**

This notebook walks through a complete data cleaning pipeline on a realistic e-commerce dataset.
Each section explains what the problem is, how we fix it, and — most importantly — *why* that strategy was chosen.

---

### Contents
1. [Load & First Look](#1)
2. [Fix Column Names](#2)
3. [Remove Duplicate Orders](#3)
4. [Standardise Categorical Columns](#4)
5. [Parse & Validate Dates](#5)
6. [Handle Missing Values](#6)
7. [Fix Data-Type Issues](#7)
8. [Detect & Cap Age Outliers](#8)
9. [Validate Email Addresses](#9)
10. [Normalise Currency → KES](#10)
11. [Export & Summary Report](#11)


## Setup

In [ ]:
import re
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)

RAW_PATH   = "messy_data__1_.csv"
CLEAN_PATH = "orders_clean.csv"

---
<a id='1'></a>
## 1. Load & First Look

We load everything as `dtype=str` to prevent pandas from guessing types.
Without this, phone numbers like `254797291993` become floats (`2.547e+11`) and lose their leading digits.


In [ ]:
df = pd.read_csv(RAW_PATH, dtype=str)
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# How many values are missing per column?
df.isnull().sum()

---
<a id='2'></a>
## 2. Fix Column Names

**Problem:** Columns have leading/trailing spaces, mixed casing, and special characters
like `(`, `)`, `/`, `%`, `-`, `?`.

**Solution:** A regex pipeline normalises everything to `snake_case`, then we apply
an explicit rename map for clarity.


In [ ]:
print("BEFORE:", list(df.columns))

In [ ]:
# Auto-clean: strip → lowercase → symbols to underscores → collapse doubles → strip trailing _
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[\s/\-\.\(\)\?%]+", "_", regex=True)
      .str.replace(r"_+", "_", regex=True)
      .str.strip("_")
)

RENAME = {
    "e_mail_address":          "email",
    "phone_no":                "phone",
    "city_town":               "city",
    "age_yrs":                 "age",
    "product_cat":             "category",
    "unit_price_kes_usd_eur":  "unit_price",
    "discount":                "discount_pct",
    "curr":                    "currency",
    "review_rating_1_5":       "review_rating",
    "full_address_street_zip": "address",
}

df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns}, inplace=True)
print("AFTER:", list(df.columns))

---
<a id='3'></a>
## 3. Remove Duplicate Orders

**Problem:** 115 rows share an `order_id` with another row.

**Strategy:** Keep the first occurrence (`keep='first'`), which is the earliest
as-loaded record and most likely the authoritative one.

> Duplicates inflate revenue totals and distort any per-order analysis.


In [ ]:
before = len(df)
df.drop_duplicates(subset="order_id", keep="first", inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Removed {before - len(df)} duplicates → {len(df)} rows remain")

---
<a id='4'></a>
## 4. Standardise Categorical Columns

**Problem:** Free-text entry created dozens of variants for fields that should
have a small, fixed set of values.

| Column | Raw variants | After |
|---|---|---|
| Order Status | 17 | 6 |
| Payment Method | 17 | 4 |
| Product Category | 24 | 7 |
| City | Many | 9 normalised |

**Solution:** A `make_mapper` factory function builds a reverse lookup dictionary
from a `{canonical: [variants]}` map and returns a function we can pass to `.apply()`.


In [ ]:
def make_mapper(mapping_dict):
    """
    Factory that turns a {canonical: [variants]} dict into an .apply()-ready function.
    
    It builds a REVERSE lookup:  variant → canonical
    Then the returned mapper function looks up each cell value in that reverse dict.
    
    Example:
        make_mapper({"pending": ["Pending", "PENDING"]})
        mapper("PENDING") → "pending"
        mapper("unknown") → "other"
        mapper(None)      → NaN
    """
    reverse = {variant: canonical
               for canonical, variants in mapping_dict.items()
               for variant in variants}

    def mapper(val):
        if pd.isna(val):
            return np.nan
        return reverse.get(str(val).strip(), "other")

    return mapper

In [ ]:
# ── Order Status ──────────────────────────────────────────────────────────────
STATUS_MAP = {
    "pending":    ["pending", "Pending", "PENDING"],
    "shipped":    ["shipped", "Shipped"],
    "delivered":  ["delivered", "Delivered", "DELIVERED", "Complete", "complete"],
    "cancelled":  ["cancelled", "Cancelled", "CANCELLED", "Canceled"],
    "returned":   ["returned", "Returned", "Return", "return"],
    "in transit": ["In Transit", "in transit"],
}
df["order_status"] = df["order_status"].apply(make_mapper(STATUS_MAP))

# ── Payment Method ────────────────────────────────────────────────────────────
PAYMENT_MAP = {
    "cash":          ["cash", "CASH", "Cash"],
    "mpesa":         ["M-PESA", "M-Pesa", "Mpesa", "mpesa", "mobile money", "Mobile Money"],
    "card":          ["card", "Card", "CARD", "Debit Card", "Credit card", "Credit Card"],
    "bank transfer": ["Bank", "bank", "Bank Transfer", "bank transfer", "BANK TRANSFER"],
}
df["payment_method"] = df["payment_method"].apply(make_mapper(PAYMENT_MAP))

# ── Product Category ──────────────────────────────────────────────────────────
CATEGORY_MAP = {
    "beauty":      ["beauty", "Beauty", "Beauti"],
    "sports":      ["sports", "Sports", "Sport"],
    "books":       ["books", "Books", "Book"],
    "home":        ["HOME", "Home & Kitchen", "Home/Kitchen", "home and kitchen", "home"],
    "clothing":    ["Clothing", "clothing", "Fashion", "fashion", "Fashn"],
    "electronics": ["Electronics", "electronics", "Electrnics", "ELEC"],
    "groceries":   ["groceries", "Groceries", "Grocery"],
}
df["category"] = df["category"].apply(make_mapper(CATEGORY_MAP))

print("Order Status:", sorted(df["order_status"].dropna().unique()))
print("Payment:     ", sorted(df["payment_method"].dropna().unique()))
print("Category:    ", sorted(df["category"].dropna().unique()))

In [ ]:
# ── City / Town ───────────────────────────────────────────────────────────────
# Custom mapper: unknown cities are kept (lowercased), not labelled "other"
CITY_MAP = {
    "nairobi":       ["nairobi", "Nairobi", "NBO", "Nairobii"],
    "mombasa":       ["mombasa", "Mombasa"],
    "kisumu":        ["kisumu", "Kisumu", "KSM"],
    "kampala":       ["kampala", "Kampala", "KLA"],
    "machakos":      ["machakos", "Machakos"],
    "thika":         ["thika", "Thika", " Thika"],
    "dar es salaam": ["dar es salaam", "Dar es Salaam", "Dar Es Salaam"],
    "nakuru":        ["nakuru", "Nakuru"],
    "eldoret":       ["eldoret", "Eldoret"],
}

def map_city(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    for canonical, variants in CITY_MAP.items():
        if v in variants:
            return canonical
    return v.lower()

df["city"] = df["city"].apply(map_city)
df["city"].value_counts().head(10)

---
<a id='5'></a>
## 5. Parse & Validate Dates

**Problem:** Five different date formats appear in the same column.
`pd.to_datetime(infer=True)` can silently mis-parse ambiguous formats —
`10-08-2024` could be Aug 10 or Oct 8 depending on locale.

**Solution:** Try each format explicitly and in order. Any value that fails all
formats becomes `NaT`.

We also flag the **418 rows** where `ship_date < order_date` — physically
impossible — and nullify the ship date rather than dropping the order.


In [ ]:
DATE_FORMATS = [
    "%d/%m/%Y",   # 21/03/2025
    "%Y-%m-%d",   # 2024-05-13
    "%d-%m-%Y",   # 10-08-2024
    "%m-%d-%Y",   # 02-26-2023
    "%d-%b-%Y",   # 20-Aug-2024
]

def parse_date(val):
    """Try each format in order; return NaT if none match."""
    if pd.isna(val) or str(val).strip() == "":
        return pd.NaT
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(str(val).strip(), fmt).date()
        except ValueError:
            continue
    return pd.NaT

for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col].apply(parse_date))
    print(f"{col}: {df[col].isna().sum()} unparseable | {df[col].min().date()} → {df[col].max().date()}")

# Nullify ship dates that precede the order date
mask_bad = (df["ship_date"] < df["order_date"]) & df["ship_date"].notna() & df["order_date"].notna()
print(f"\nship_date < order_date: {mask_bad.sum()} rows → nullified")
df.loc[mask_bad, "ship_date"] = pd.NaT

---
<a id='6'></a>
## 6. Handle Missing Values

**Key insight:** Not all NaNs are equal. The *reason* for missingness determines the right strategy.

| Column | Strategy | Why |
|---|---|---|
| `email` | Sentinel fill | Row is useful; flag for CRM follow-up |
| `phone` | Sentinel fill | Same |
| `age` | Median imputation | Continuous; median is robust to outliers |
| `order_date` | Drop row | Date-less orders are unanalysable |
| `ship_date` | Keep NaT | NaT = "not yet shipped" — valid state |
| `review_rating` | Keep NaN | Customer chose not to review |
| `marketing_source` | Fill "Unknown" | Preserves row; signals gap |
| `address` | Fill placeholder | Preserves row; flags for ops |


In [ ]:
# Cast numeric columns before imputation
for col in ["age", "unit_price", "total_amount", "qty", "discount_pct", "review_rating"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["email"]  = df["email"].fillna("no_email@unknown.com")
df["phone"]  = df["phone"].fillna("000000000000")
df["age"]    = df["age"].fillna(df["age"].median())

n_before = len(df)
df.dropna(subset=["order_date"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Dropped {n_before - len(df)} rows with no order_date")

df["marketing_source"] = df["marketing_source"].fillna("Unknown")
df["address"]          = df["address"].fillna("Address not provided")

print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

---
<a id='7'></a>
## 7. Fix Data-Type Issues

**Problem:** Phone numbers were stored as floats in scientific notation.
All columns need to be cast to their correct Python types.

**Key technique:** `pd.to_numeric(errors='coerce')` converts cleanly or silently
produces NaN — safer than direct casting which raises errors on bad values.


In [ ]:
# Phone: strip float artefact '.0', then zero-pad to 12 characters
# zfill(12) adds leading zeros: '123' → '000000000123'
df["phone"] = (
    df["phone"]
      .str.replace(r"\.0$", "", regex=True)
      .str.strip()
      .str.zfill(12)
)

df["currency"] = df["currency"].str.strip().str.upper()

for col in ["unit_price", "total_amount", "discount_pct", "review_rating"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["qty", "age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("Phone sample:", df["phone"].head(3).tolist())
df.dtypes

---
<a id='8'></a>
## 8. Detect & Cap Age Outliers

**Problem:** Ages of **-4, 5, 12, and 130** appear in the data (39 rows total).

**Strategy:** Replace with the median of *valid* ages (those in [16, 100]).
We don't drop the rows — only the age field is wrong; the rest of the order is valid.


In [ ]:
AGE_MIN, AGE_MAX = 16, 100

valid_mask   = df["age"].between(AGE_MIN, AGE_MAX)
median_age   = int(df.loc[valid_mask, "age"].median())
outlier_mask = ~valid_mask

print(f"Outliers: {outlier_mask.sum()} rows")
print(f"Values:   {sorted(df.loc[outlier_mask, 'age'].unique().tolist())}")
print(f"Replacing with median: {median_age}")

df.loc[outlier_mask, "age"] = median_age
df["age"].describe()

---
<a id='9'></a>
## 9. Validate Email Addresses

**Problem:** 218 emails were missing; others may be malformed.

**Strategy:** Regex validation adds a boolean `email_valid` column — non-destructive.
The analyst (or CRM team) can then filter on this flag rather than having rows silently dropped.

**Regex pattern:** `^[\w.%+\-]+@[\w.\-]+\.[a-zA-Z]{2,}$`


In [ ]:
EMAIL_RE = re.compile(r"^[\w.%+\-]+@[\w.\-]+\.[a-zA-Z]{2,}$")
SENTINEL_EMAILS = {"no_email@unknown.com", "nan", ""}

def is_valid_email(val):
    s = str(val).strip()
    if s in SENTINEL_EMAILS:
        return False
    return bool(EMAIL_RE.match(s))

df["email_valid"] = df["email"].apply(is_valid_email)

print(f"Valid emails  : {df['email_valid'].sum()}")
print(f"Invalid/missing: {(~df['email_valid']).sum()} ({(~df['email_valid']).mean()*100:.1f}%)")

---
<a id='10'></a>
## 10. Normalise Currency → KES

**Problem:** `total_amount` contains values in KES, USD, and EUR.
Comparing or summing them directly is meaningless.

**Strategy:** Add a new `total_kes` column. Keep the original columns — converting
in-place would destroy information about what currency was actually used.

> In production, pull live FX rates from an API and log the retrieval date
> so results are reproducible.


In [ ]:
FX_TO_KES = {"KES": 1.0, "USD": 129.5, "EUR": 140.2}

# axis=1 means apply row-by-row (across columns), not column-by-column
df["total_kes"] = df.apply(
    lambda row: row["total_amount"] * FX_TO_KES.get(row["currency"], np.nan),
    axis=1
)

print("Currency distribution:")
print(df["currency"].value_counts())
print()
df["total_kes"].describe().round(2)

---
<a id='11'></a>
## 11. Export & Summary Report


In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Saved → {CLEAN_PATH}  |  Shape: {df.shape}")

In [ ]:
# Before vs After summary
summary = {
    "Metric": [
        "Total rows", "Columns",
        "Duplicate orders removed",
        "Order Status variants",
        "Payment Method variants",
        "Product Category variants",
        "Age outliers fixed",
        "Impossible ship dates nullified",
        "Currencies unified (new column)",
    ],
    "Raw": [5115, 21, 115, 17, 17, 24, 39, 418, "3 mixed"],
    "Clean": [len(df), len(df.columns), 0, 6, 4, 7, 0, 0, "3 + KES col"],
}
pd.DataFrame(summary)

In [ ]:
# Final peek at the clean data
df[["order_id","customer_name","city","category","order_status",
    "payment_method","total_kes","email_valid"]].head(8)

---
## 💡 What I Learned

**1. Every null is not the same null.**
A missing `ship_date` means the order hasn't shipped yet. A missing `review_rating` means the customer didn't review. A missing `order_date` means the row is unanalysable. Applying a blanket `.dropna()` or `.fillna(0)` would have destroyed real signal.

**2. Load data defensively.**
Reading everything as `dtype=str` first, then casting explicitly column by column, prevents pandas from silently corrupting identifiers like phone numbers.

**3. The reason behind a cleaning decision matters as much as the decision itself.**
Dropping vs imputing vs flagging all produce valid-looking outputs. What separates strong analysis is being able to articulate *why* — and documenting it so the next person doesn't have to guess.

**4. Non-destructive changes preserve optionality.**
Adding `total_kes` instead of overwriting `total_amount`, and adding `email_valid` instead of dropping bad emails, keeps original data intact and lets downstream analysts make their own decisions.

**5. Standardisation before analysis is non-negotiable.**
`groupby('order_status')` on the raw data returns 17 groups. After cleaning: 6. Every chart or aggregation built on uncleaned categoricals would have been silently wrong.

**6. Factory functions keep mapping logic reusable and auditable.**
The `make_mapper` function handles Order Status, Payment Method, and Product Category all the same way. Adding a new category variant means editing one dictionary — not hunting through scattered `if/elif` chains.

**7. Cleaning is iterative, not linear.**
Several issues only became visible after fixing earlier ones. The age outliers were masked until the column was cast to numeric. Real cleaning requires going back and forth, not just running a checklist once.
